# Chapter 05-12 · The applied checkpoint: one dataset, end to end

**No new algorithms in this chapter.** Everything here was taught in 05-01 to 05-11. What is new is that
nobody tells you which one to use, the data has real defects, and the answer at the end is not a single
number.

---

## Before you read: the shadow pass

**Four minutes, and it will feel like it is not working. That feeling is the method.**

**Step 1 - scan the map, and only the map.** Below is every heading in this chapter. Read the list. Do
not scroll past it, do not highlight, do not take notes.

> 1. The prediction contract
> 2. The data, and the split that comes before looking at it
> 3. What this dataset is not
> 4. The ladder: five models, one honest comparison
> 5. Opening the test set, once
> 6. The number that hides six other numbers
> 7. Absolute error is flat; relative error is not
> 8. Failure lab: just delete the bad rows
> 9. What to write in the report

**Step 2 - close this notebook and write what each heading means.** From memory, on paper, in your own
words. One or two sentences each. You have not read the chapter, so most of it will be guesswork built
out of 05-01 to 05-11 - **that is exactly what is wanted.** Guess. Be wrong in specific ways.

**Step 3 - come back and mark your gaps as you read.** Where the chapter contradicts what you wrote, mark
it. Those marks are worth more than any highlighting you could have done.

<details>
<summary><b>Only after you have written something - a one-line gloss of each heading, to check against</b></summary>

<br>

1. Deciding what is being predicted, at what moment, from what - before touching a model.
2. Splitting into fit / watch / test *first*, so nothing you see can leak into a choice.
3. The recorded values stop at a ceiling, and two columns are not what their names suggest.
4. Baseline, linear, tree, forest, boosting - each one earning its place over the one below.
5. One measurement, at the end, on data no choice was made against.
6. The overall error, broken apart by segment, where one segment is twice as bad as the rest.
7. The same absolute error means something very different for a cheap area and an expensive one.
8. Deleting the awkward rows makes the metric worse, not better, and hurts most where it was meant to help.
9. What an honest write-up contains, including the parts that are not flattering.

</details>

**Why this works, briefly.** Trying to retrieve something you have not learnt yet still improves how well
you learn it - the effect is called *prequestioning*, and it holds even when every guess is wrong.
Generating an answer beats reading one (*the generation effect*), and the difficulty is doing the work:
Bjork's term for it is a **desirable difficulty**. Rereading feels much better and does much less.

## Warm-up: retrieve across the whole module, not just the last chapter

Answer from memory before opening anything. These deliberately jump around - **04-05 and 05-04 are as
likely to be tested as 05-11**, because that mixing is what makes retrieval hold.

1. *(04-05)* Why must the test set be split off **before** you look at the data?
2. *(05-04)* RMSE and MAE disagree about which model is better. What does that tell you about the errors?
3. *(05-05)* You plot residuals against the prediction and see a fan opening to the right. What is wrong?
4. *(05-08)* Training error 0.19, held-out error 0.53. Bias problem or variance problem?
5. *(05-11)* Why is `n_estimators` a hyperparameter for boosting but not for a forest?

<details>
<summary><b>Answers</b></summary>

<br>

1. Because anything you learn from looking - which columns matter, which rows are odd, which model to try
   - becomes a decision made *using* the test set, and the test set stops being an estimate of unseen
   performance. It becomes a number you tuned.
2. That a few large errors are doing the work. RMSE squares them, MAE does not, so they rank differently
   only when the error distribution has a heavy tail.
3. The spread of the errors grows with the size of the prediction - heteroscedasticity. Often a sign that
   the target should be modelled on a log scale, or that the error is proportional rather than absolute.
4. Variance. The gap is the symptom; a bias problem has both numbers high and close together.
5. Because a forest averages independent trees, so more of them can only sharpen the average, while
   boosting *sums* corrections and eventually starts correcting noise.

</details>

## Why this matters

Every chapter so far handed you a question. This one hands you a table.

**The gap between "can fit a model" and "can be trusted with a dataset" is almost entirely in the parts
that are not modelling** - deciding what is being predicted, splitting before looking, noticing that the
data has a ceiling, and refusing to report one number when one number is a lie.

By the end you will have a model that scores **0.4574 RMSE** on data it has never seen, and you will be
able to say the more useful thing: *it scores 0.4256 on 96% of the rows and 0.9068 on the other 4%, in one
direction, for a reason that no model can fix.*

## What you will be able to do

- Fill in a prediction contract for a dataset you have not seen before
- Split three ways and say what each part is allowed to be used for
- Compare five models honestly and pick one without touching the test set
- Break an error down by segment and find where the model is quietly failing
- Recognise a censored target, and say why deleting it makes things worse
- Write the paragraph a colleague actually needs, caveats included

## The prediction contract

04-04 introduced this and it has not changed: **answer these before a model exists.**

<img src="../../assets/05_regression/05-12/three_way_split.svg" alt="The 20,640 rows split into fit, watch and test; the fit and watch sets used in a loop while the test set stays sealed until one final measurement." width="860">

| Question | Answer for this chapter |
|---|---|
| **What is one row?** | One census block group - the smallest area the US Census publishes sample data for, typically 600-3,000 people. **Not one house.** |
| **What are we predicting?** | The **median** house value in that block group, in units of 100,000 dollars |
| **Explaining, nowcasting, or forecasting?** | **Explaining.** Every column is measured in the same census as the target. Nothing here predicts the future |
| **What would be known at prediction time?** | In a real deployment: nothing, until the next census. This is a *cross-sectional* exercise, and pretending otherwise would be the leak 04-06 warned about |
| **What does a mistake cost?** | Unstated - so we report both RMSE and MAE, and we report them by segment |

**That third row is the honest one and it is easy to skip.** This dataset cannot be used to predict a
future house price. It can be used to ask "given what the census recorded about an area, what was the
median value there" - which is a real question, and a different one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# Real data. Source, licence and caveats: data/README.md, section 3a.
# First call downloads about 360 KB and caches it; later calls are offline.
frame = fetch_california_housing(as_frame=True).frame
FEATURES = [column for column in frame.columns if column != "MedHouseVal"]
TARGET = "MedHouseVal"

print("%d rows, %d feature columns" % (len(frame), len(FEATURES)))
print(FEATURES)

## The data, and the split that comes before looking at it

**The split happens now**, before a single histogram. Three parts, and each one has a job it is not
allowed to leave:

- **fit** - the model learns from these rows.
- **watch** - every choice is scored here. Which model, which depth, how many rounds.
- **test** - opened once, at the end, and never used to change anything.

Watch the animation above again if the reason is not obvious: the loop between the blue and green boxes
runs as many times as you like, and each lap costs you nothing. The red box has exactly one use.

In [ ]:
train, test = train_test_split(frame, test_size=0.2, random_state=0)
fit, watch = train_test_split(train, test_size=0.25, random_state=0)

print("fit   %6d rows   the model learns from these" % len(fit))
print("watch %6d rows   every decision is scored here" % len(watch))
print("test  %6d rows   sealed until the end" % len(test))


def rmse(model, part):
    return float(np.sqrt(((part[TARGET] - model.predict(part[FEATURES])) ** 2).mean()))


def mae(model, part):
    return float(np.abs(part[TARGET] - model.predict(part[FEATURES])).mean())

## What this dataset is not

Now we look - **at the training rows only.** The test set stays shut, and `watch` is for scoring, not for
browsing. Everything below is learnt from `train`.

### Predict before running

The target is a median house value in 100,000s of dollars, from the 1990 census. Sketch the shape of its
distribution before you look. Where do you expect the extremes?

In [ ]:
CEILING = frame[TARGET].max()
ceiling_rows = int((train[TARGET] >= CEILING - 1e-9).sum())
just_below = int(((train[TARGET] >= 4.9) & (train[TARGET] < 5.0)).sum())
whole_band = int(((train[TARGET] >= 4.0) & (train[TARGET] < 4.9)).sum())

print("highest value anywhere in the data : %.5f" % CEILING)
print("training rows sitting exactly there: %d  (%.2f%% of the training set)"
      % (ceiling_rows, 100 * ceiling_rows / len(train)))
print("training rows from 4.9 up to 5.0   : %d" % just_below)
print("training rows from 4.0 up to 4.9   : %d" % whole_band)
print("lowest value                       : %.5f, on %d training rows"
      % (train[TARGET].min(), int((train[TARGET] <= train[TARGET].min() + 1e-9).sum())))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.3))

left.hist(train[TARGET], bins=100, color="#b0b0b0")
left.axvline(CEILING, color="#c0392b", lw=1.8)
left.annotate("%d rows stacked on one value" % ceiling_rows, xy=(CEILING, ceiling_rows * 0.75),
              xytext=(2.5, ceiling_rows * 0.82), fontsize=10, color="#c0392b", ha="center",
              bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none"),
              arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.3))
left.set_xlabel("median house value (100,000s of dollars)")
left.set_ylabel("training rows")
left.set_title("the target, as recorded")

zoom = train[(train[TARGET] >= 3.8)]
right.hist(zoom[TARGET], bins=60, color="#b0b0b0")
right.axvline(CEILING, color="#c0392b", lw=1.8)
right.set_xlabel("median house value (100,000s of dollars)")
right.set_ylabel("training rows")
right.set_title("the top of the range, magnified")
right.annotate("%d rows here" % ceiling_rows, xy=(CEILING, ceiling_rows * 0.9),
               xytext=(4.25, ceiling_rows * 0.72), fontsize=10, color="#c0392b", ha="center",
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none"),
               arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.3))
right.annotate("%d rows in this\nwhole tenth" % just_below, xy=(4.94, 14),
               xytext=(4.42, ceiling_rows * 0.30), fontsize=10, color="#333333", ha="center",
               arrowprops=dict(arrowstyle="->", color="#333333", lw=1.2))

fig.suptitle("Two views of the same column, and only one of them shows the problem", y=1.02)
fig.tight_layout()
plt.show()

**784 training rows sit on a single value, while the entire tenth just below it holds 32.**

That is not a housing market. **It is a recording rule.** Whoever assembled this file capped the median
value at 500,001 dollars, and every block group worth more than that was written down as worth exactly
that. The information above the line was never collected.

**Three consequences, and they run through the rest of the chapter:**

1. **No model can predict those rows correctly**, because their true values are not in the data. The best
   any model can do is guess the ceiling for all of them, which is wrong for every one that was worth more.
2. **The error on those rows will be one-directional.** A model trained mostly on cheaper areas will
   under-call them, systematically, and no symmetric metric will tell you which direction.
3. **The overall RMSE will look fine anyway**, because they are only 4.7% of the rows. This is the exact
   shape of a problem that segment analysis exists to find.

There is a floor too - three training rows at 0.15, the lowest value in the file - but three rows is a
curiosity rather than a problem.

In [ ]:
odd = pd.DataFrame([
    {"column": "AveRooms", "what it sounds like": "rooms in a house",
     "what it is": "total rooms / total households",
     "max in training": train["AveRooms"].max(),
     "implausible above": 20, "rows above that": int((train["AveRooms"] > 20).sum())},
    {"column": "AveOccup", "what it sounds like": "people in a household",
     "what it is": "population / total households",
     "max in training": train["AveOccup"].max(),
     "implausible above": 10, "rows above that": int((train["AveOccup"] > 10).sum())},
])
print(odd.to_string(index=False, float_format=lambda v: "%.1f" % v))

worst = train.nlargest(3, "AveOccup")[["Population", "AveOccup", "AveRooms", TARGET]]
print("\nthe three most extreme AveOccup rows in the training set:")
print(worst.to_string(float_format=lambda v: "%.2f" % v))

**A block group with an average household size of 599 people is not a neighbourhood of very large
families.** It is a barracks, a prison, a university dormitory - somewhere the census counted thousands of
residents and very few households, so a per-household average stops meaning anything.

**We are not deleting them.** They are real places and they will appear in any future data too. What
matters is knowing they are there, so that when the model does something strange on them we recognise the
cause instead of blaming the algorithm. 02-08's rule stands: **a column is what it was computed from, not
what its name suggests.**

## The ladder: five models, one honest comparison

Every model below has already been taught. The point is the *procedure*: start with the cheapest thing
that could work, and make each step earn its place on the `watch` set.

### Predict before running

Rank these five by held-out error before you run the cell. Then check how many you got right.

| | |
|---|---|
| the mean of the training target | 05-01 |
| linear regression | 05-03 |
| a single tree, depth 8 | 05-10 |
| a forest of 200 trees | 05-10 |
| boosting, 400 rounds | 05-11 |

In [ ]:
ladder = [
    ("the mean of the training target", DummyRegressor(strategy="mean")),
    ("linear regression", LinearRegression()),
    ("one tree, depth 8", DecisionTreeRegressor(max_depth=8, random_state=0)),
    ("a forest of 200 trees", RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)),
    ("boosting, 400 rounds", HistGradientBoostingRegressor(max_iter=400, learning_rate=0.1,
                                                           random_state=0)),
]

rows, trained = [], {}
for label, model in ladder:
    model.fit(fit[FEATURES], fit[TARGET])
    trained[label] = model
    rows.append({"model": label, "fit RMSE": rmse(model, fit),
                 "watch RMSE": rmse(model, watch), "watch MAE": mae(model, watch)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

**The order is the one you would hope for, and two of the numbers are worth stopping on.**

**Linear regression cuts the baseline's error by a third** (1.1620 to 0.7493) from eight columns and no
tuning. That is the number every later model has to beat to justify itself, and in a real project it is
often the model that ships.

**The forest's fit RMSE is 0.1925 against a watch RMSE of 0.5283.** That gap is not a bug - 05-10 showed
that a fully grown forest memorises its training data almost perfectly and still generalises, because the
averaging is what does the work. **Do not read a training error on a forest as evidence of anything.**

**Boosting wins on both metrics**, 0.4705 RMSE and 0.3060 MAE. RMSE and MAE agreeing on the ranking is
mild evidence that no single group of rows is dominating the comparison - 05-04's check, and it passes.

<img src="../../assets/05_regression/05-12/model_ladder.gif" alt="Predicted against actual for each of the five models in turn: the grey cloud tightens onto the diagonal while the red row of capped points stays spread out along the ceiling." width="620">

**Watch the grey cloud collapse onto the diagonal - and watch the red points refuse to.** Those are the
rows recorded at the ceiling. Under the mean model they sit in a vertical line like everything else; by
the time boosting has finished they are still smeared across predictions from 1.2 to 5.2, because there
is nothing in the columns that says *how far above the ceiling* a block group really was.

**Five models, each better than the last, and not one of them improves that row.** That is the chapter in
one image: the ladder fixes what is fixable, and the segment analysis finds what is not.

## Opening the test set, once

Boosting won on `watch`, so boosting is what ships. **The choice is made. Nothing after this point may
change it** - if the test number disappoints and we go back and try something else, the test set has
quietly become a second watch set and its estimate is no longer honest.

In [ ]:
chosen = trained["boosting, 400 rounds"]

print("watch  RMSE %.4f   MAE %.4f   <- what we chose on" % (rmse(chosen, watch), mae(chosen, watch)))
print("TEST   RMSE %.4f   MAE %.4f   <- the estimate we report"
      % (rmse(chosen, test), mae(chosen, test)))

**0.4574 on the test set against 0.4705 on watch.** The test number came out slightly *better*, which
happens - these are two finite samples and the difference is well inside the noise. If it had come out
much worse, the honest response would be to report it and say why, not to go looking for a better model.

In the units of the data, an RMSE of 0.4574 is about **45,700 dollars** of typical error on a median value
that averages 207,000. **And that sentence is where most write-ups stop.**

## The number that hides six other numbers

**Before any grouping, look at the errors the way 05-05 taught.**

In [ ]:
scored = test.copy()
scored["prediction"] = chosen.predict(test[FEATURES])
scored["error"] = scored["prediction"] - scored[TARGET]
on_ceiling = (scored[TARGET] >= CEILING - 1e-9).to_numpy()

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6))

left.scatter(scored["prediction"][~on_ceiling], scored["error"][~on_ceiling],
             s=7, color="#b0b0b0", label="ordinary rows")
left.scatter(scored["prediction"][on_ceiling], scored["error"][on_ceiling],
             s=14, color="#c0392b", label="recorded at the ceiling")
left.axhline(0, color="#333333", lw=1.2)
left.set_xlabel("prediction")
left.set_ylabel("error  (prediction - actual)")
left.set_title("every test row")
left.legend(loc="upper left", fontsize=9)
left.annotate("this straight edge is the ceiling:\nno row above 5 exists to be wrong about",
              xy=(4.6, -0.35), xytext=(1.35, -3.0), fontsize=9, color="#c0392b",
              bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none"),
              arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.2))

right.hist(scored["error"][~on_ceiling], bins=60, color="#b0b0b0", label="ordinary rows")
right.hist(scored["error"][on_ceiling], bins=30, color="#c0392b", label="at the ceiling")
right.axvline(0, color="#333333", lw=1.2)
right.axvline(scored["error"][on_ceiling].mean(), color="#c0392b", lw=1.8, ls="--")
right.text(scored["error"][on_ceiling].mean() - 0.12, 200, "their mean error, %.4f"
           % scored["error"][on_ceiling].mean(), fontsize=9, color="#c0392b", ha="right")
right.set_yscale("log")
right.set_xlabel("error  (prediction - actual)")
right.set_ylabel("test rows (log scale)")
right.set_title("the same errors, as two distributions")
right.legend(loc="upper left", fontsize=9)

fig.suptitle("One model, and one group of rows behaving differently from the rest", y=1.02)
fig.tight_layout()
plt.show()

**The left panel has a straight edge on it, and straight edges in a residual plot are never natural.**
Every red point sits on a line running down and to the right: as the prediction rises, the error on a
capped row falls, because the actual value is pinned at exactly 5.0 for all of them. Nothing about the
housing market produces a line like that. A recording rule does.

**The right panel is the same fact as two shapes.** The grey distribution is roughly centred on zero,
which is what an unbiased model looks like. The red one sits almost entirely to the left of it - the model
under-calls nearly every capped row, and by a lot.

**05-05's lesson applies unchanged:** the residuals carry the structure the metric threw away. What is new
here is that the structure points at the *data* rather than at the model.

In [ ]:
def by_segment(column):
    # RMSE, bias and size for each group - the three numbers a segment needs
    out = []
    for name, group in scored.groupby(column, observed=True):
        out.append({str(column): name, "rows": len(group),
                    "RMSE": float(np.sqrt((group["error"] ** 2).mean())),
                    "mean error (+ = too high)": float(group["error"].mean()),
                    "mean actual": float(group[TARGET].mean())})
    return pd.DataFrame(out)


scored["at the ceiling"] = np.where(scored[TARGET] >= CEILING - 1e-9, "yes", "no")
print("overall test RMSE %.4f\n" % rmse(chosen, test))
print(by_segment("at the ceiling").to_string(index=False, float_format=lambda v: "%.4f" % v))

**0.9068 against 0.4256. The model is more than twice as bad on the capped rows, and it is wrong in one
direction: on average it calls them 0.5509 too low.**

That is 55,000 dollars of systematic under-valuation on 4.4% of the test set, and **the overall RMSE of
0.4574 contains it without showing it.** A stakeholder reading only that number would have no way to know.

**The cause is not the model, and no amount of tuning will help.** The rows were recorded at a ceiling;
their true values are missing from the data. What the model does is entirely reasonable given what it was
shown - it just cannot be right.

**What you do about it is a reporting decision, not a modelling one.** Say the model is unreliable above
roughly 450,000 dollars. Or predict a floor rather than a value for those areas. Or find a data source
without the cap. What you do not do is quote 0.4574 and stop.

In [ ]:
scored["income band"] = pd.qcut(scored["MedInc"], 5,
                                labels=["lowest", "low", "middle", "high", "highest"])
bands = by_segment("income band")
bands["RMSE as % of the mean value"] = 100 * bands["RMSE"] / bands["mean actual"]
print(bands.to_string(index=False, float_format=lambda v: "%.4f" % v))

<img src="../../assets/05_regression/05-12/segment_split.gif" alt="A single RMSE bar of 0.4574 separating into seven bars: the capped segment rises to 0.907 while every other segment stays near the overall line." width="820">

## Absolute error is flat; relative error is not

**Read the RMSE column first: 0.4646, 0.4282, 0.4678, 0.4453, 0.4797.** Those are all the same number.
Split by income, the model looks perfectly even-handed - and if you stopped here you would report that it
treats rich and poor areas alike.

**Now read the last column: 38.45%, 26.94%, 24.48%, 19.51%, 14.64%.**

The same absolute error is **two and a half times larger as a share of the value** in the cheapest areas
as in the most expensive. Being 46,000 dollars out on a 121,000-dollar block group and being 48,000 out on
a 328,000-dollar one are not the same mistake, and only one of the two columns says so.

**This is 05-04's argument arriving in a real decision.** RMSE and a percentage error encode different
beliefs about what a mistake costs. The chapter warned that MAPE has genuine problems - it explodes near
zero and it punishes over-prediction more than under-prediction - and none of that makes the absolute
number automatically correct. **Which one is right here depends on who is harmed by the error**, and that
is a question about the application, not about the arithmetic.

**Two segmentations, two completely different verdicts, one model.** Neither is a trick. Both are true.
That is why "report error by segment" means *by several segments*, chosen because someone would act on
them - not one convenient split.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.3), sharex=True)
order = ["lowest", "low", "middle", "high", "highest"]
positions = np.arange(len(order))

left.bar(positions, bands["RMSE"], 0.6, color="#2c5f9e")
left.axhline(rmse(chosen, test), color="#333333", lw=1.2, ls=":")
left.text(2.0, 0.512, "the overall number, %.4f" % rmse(chosen, test), fontsize=9,
          color="#333333", ha="center")
for x, value in zip(positions, bands["RMSE"]):
    left.text(x, value + 0.008, "%.3f" % value, ha="center", fontsize=9, color="#2c5f9e")
left.set_ylim(0, 0.58)
left.set_ylabel("RMSE, in 100,000s of dollars")
left.set_title("absolute error: the same everywhere")

left.set_xticks(positions)
left.set_xticklabels(order)
right.bar(positions, bands["RMSE as % of the mean value"], 0.6, color="#c0392b")
for x, value in zip(positions, bands["RMSE as % of the mean value"]):
    right.text(x, value + 0.7, "%.1f%%" % value, ha="center", fontsize=9, color="#c0392b")
right.set_ylim(0, 46)
right.set_ylabel("RMSE as a share of the band's mean value")
right.set_title("relative error: two and a half times worse at the bottom")
right.set_xticks(positions)
right.set_xticklabels(order)

for axis in (left, right):
    axis.set_xlabel("block groups, by median income of the area")

fig.suptitle("The same five numbers, divided by the value being predicted", y=1.02)
fig.tight_layout()
plt.show()

**The left chart is why "the model is fair across income levels" gets said, and the right chart is why it
is not true.** Only the denominator changed.

**Neither chart is the correct one.** Which of them describes the harm depends entirely on the decision
downstream. If the model feeds a total valuation for a portfolio, dollars are what matter and the left
chart is right. If it feeds a per-area assessment that a household appeals against, being 38% out is far
worse than being 15% out, and the right chart is right. **The analyst's job is to produce both and to say
which question each one answers.**

In [ ]:
scored["age band"] = pd.cut(scored["HouseAge"], [0, 15, 30, 45, 52],
                            labels=["1-15 years", "16-30", "31-45", "46-52"])
print(by_segment("age band").to_string(index=False, float_format=lambda v: "%.4f" % v))


def region_of(row):
    if row["Latitude"] > 37.0 and row["Longitude"] < -121.0:
        return "Bay Area"
    if row["Latitude"] < 34.5 and row["Longitude"] > -119.0:
        return "Greater LA / San Diego"
    return "everywhere else"


scored["region"] = scored.apply(region_of, axis=1)
print()
print(by_segment("region").to_string(index=False, float_format=lambda v: "%.4f" % v))

**Where the errors are, drawn on the state.**

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 5.6), sharex=True, sharey=True)

limit = 1.0
spots = left.scatter(scored["Longitude"], scored["Latitude"], c=scored["error"].clip(-limit, limit),
                     cmap="coolwarm", s=7, vmin=-limit, vmax=limit)
left.set_title("signed error, by location")
fig.colorbar(spots, ax=left, label="prediction - actual  (clipped at +/- 1)")

right.scatter(scored["Longitude"][~on_ceiling], scored["Latitude"][~on_ceiling],
              s=5, color="#d8d8d8", label="ordinary rows")
right.scatter(scored["Longitude"][on_ceiling], scored["Latitude"][on_ceiling],
              s=16, color="#c0392b", label="recorded at the ceiling")
right.set_title("where the capped rows are")
right.legend(loc="upper right", fontsize=9)

for axis in (left, right):
    axis.set_xlabel("longitude")
left.set_ylabel("latitude")

fig.suptitle("California, one dot per test block group", y=0.99)
fig.tight_layout()
plt.show()

**The shape of the state appears without anyone drawing it**, because latitude and longitude are two of
the eight columns - which is also why a tree-based model can carve out neighbourhoods that a linear model
cannot.

**The right panel is the useful one.** The capped rows are not scattered at random: they cluster on the
coast around the Bay Area and the Los Angeles basin, exactly where 1990 house values ran past 500,000
dollars. **So "the model is unreliable above the ceiling" and "the model is unreliable in a few specific
expensive coastal areas" are the same sentence** - and the second one is the version a stakeholder can
act on.

The left panel is deliberately reassuring by comparison: red and blue are mixed together nearly
everywhere, which is what a model with no gross spatial bias looks like. **Two panels, one finding each,
and the pair is worth more than either.**

**Two more cuts, and one of them finds nothing.** The regions come out at 0.4545, 0.4739 and 0.4075 -
close enough that the model is not obviously worse in any of them. **A segmentation that finds nothing is
a real result, and it belongs in the report**; it is how you say "we checked" rather than "we assumed".

The age bands do find something: **0.5543 on the oldest housing against 0.4094 on the 31-45 band.** The
oldest band is also the smallest (445 rows), so some of that is sampling noise - but "1-15 years" has 658
rows and sits at 0.4520, so size alone does not explain it. Worth a sentence in the report and a look at
whether the 52-year cap on `HouseAge` is doing the same thing the value ceiling did.

## Failure lab: just delete the bad rows

The capped rows are ruining the metric. **The obvious fix is to drop them from training.** Predict what
happens before running it - to the overall RMSE, to the uncapped rows, and to the capped rows themselves.

In [ ]:
fit_without = fit[fit[TARGET] < CEILING - 1e-9]
deleted = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.1, random_state=0).fit(
    fit_without[FEATURES], fit_without[TARGET])

at_ceiling = (test[TARGET] >= CEILING - 1e-9).to_numpy()
comparison = []
for label, model in [("kept the capped rows", chosen), ("deleted them", deleted)]:
    comparison.append({
        "training set": label,
        "rows fitted on": len(fit) if label.startswith("kept") else len(fit_without),
        "test RMSE (all)": rmse(model, test),
        "on the 3,947 ordinary rows": rmse(model, test[~at_ceiling]),
        "on the 181 capped rows": rmse(model, test[at_ceiling])})
print(pd.DataFrame(comparison).to_string(index=False, float_format=lambda v: "%.4f" % v))

**It gets worse. Overall 0.4574 becomes 0.4726, and on the capped rows 0.9068 becomes 1.1394.**

The one thing that improves is the ordinary rows, 0.4256 to 0.4172 - and that improvement is real. **The
trade is a small gain on 96% of the data for a large loss on exactly the 4% the deletion was supposed to
help.**

**Why it backfires:** the capped rows were the only evidence the model had that expensive areas exist at
all. Remove all 580 of them from the fitting set and the model's picture of the top of the market is built
entirely from rows below the cap, so it predicts lower everywhere near the top - which makes it *more*
wrong about the rows it can never get right.

> **The general rule, and it is worth more than this dataset:** deleting rows because they hurt your
> metric improves your metric by removing the evidence, not the problem. If a row is genuinely
> mis-recorded, deleting it is defensible and must be stated. If it is awkward but real, deleting it is
> how a model comes to be confidently wrong about the part of the world you excluded.

## Why a small bad segment can hide, exactly

This is worth doing once with algebra rather than intuition, because it tells you *how small* a segment
has to be before a metric stops noticing it.

Squared error is a **mean**, so the overall mean squared error is the segment ones weighted by size:

$$\text{MSE}_{\text{all}} \;=\; \frac{1}{N}\sum_g n_g \,\text{MSE}_g$$

and the overall RMSE is its square root. Nothing else - no correction term, no interaction.

In [ ]:
ordinary = scored[scored["at the ceiling"] == "no"]
capped = scored[scored["at the ceiling"] == "yes"]

pooled = (len(ordinary) * (ordinary["error"] ** 2).mean()
          + len(capped) * (capped["error"] ** 2).mean()) / len(scored)

print("%d ordinary rows at RMSE %.4f" % (len(ordinary), np.sqrt((ordinary["error"] ** 2).mean())))
print("%d capped   rows at RMSE %.4f" % (len(capped), np.sqrt((capped["error"] ** 2).mean())))
print("\npooled by hand : %.4f" % np.sqrt(pooled))
print("straight RMSE  : %.4f" % rmse(chosen, test))

**The same number to four decimal places, and the weights are the lesson.**

The capped rows are 4.4% of the test set. Their squared error is 0.9068² = 0.8223 against 0.4256² = 0.1811
for the rest - **four and a half times worse.** And yet they move the overall RMSE from 0.4256 to only
0.4574, because 4.4% of four-and-a-half times is a small push.

> **The rule to carry:** a segment can be arbitrarily bad and still barely move a global metric, provided
> it is small. **Global metrics are insensitive to small segments by construction, not by accident.** If
> you care about a group that is 2% of your data, no overall number will ever tell you how you are doing
> on it. You have to ask separately.

## What to write in the report

Everything above collapses into a paragraph. This is what an honest one looks like:

> A gradient boosting model predicts the median house value of a California census block group with an
> RMSE of **0.4574** (about 45,700 dollars) on 4,128 held-out rows, against **0.7493** for linear
> regression and **1.1620** for predicting the training mean. The model was chosen on a separate 4,128-row
> validation split; the test set was used once.
>
> **The error is not evenly distributed.** On the 4.4% of rows whose recorded value hits the file's
> 500,001-dollar ceiling, RMSE is **0.9068** and the model under-predicts by **0.5509** on average. This is
> a property of the data rather than the model - values above the ceiling were never recorded - and it
> cannot be fixed by fitting anything differently. **Predictions above roughly 4.5 should be read as "at
> least this much".**
>
> Absolute error is flat across income levels (0.4282 to 0.4797) but **relative** error is not: 38.4% of
> the mean value in the poorest fifth of block groups against 14.6% in the richest. Which of those matters
> depends on the decision the prediction feeds.
>
> **Scope.** The data is the 1990 US census, one state, one point in time. The unit is a block group of
> 600-3,000 people, not a house. The model explains recorded values; it does not forecast prices, and
> nothing here supports a claim about any individual property.

**Notice what that paragraph does not do.** It does not lead with the best number, it does not describe
the model's architecture, and it does not claim the model is good. It says what was measured, on what,
where it fails, and what it must not be used for.

## Common misconceptions

**"The test RMSE is the model's accuracy."** It is *an* estimate of average squared error on rows drawn
the same way as the test set. It is not a guarantee, it is not accuracy on any particular row, and as this
chapter showed, it can be a poor description of every segment it is made of.

**"The capped rows are outliers, so remove them."** An outlier is a row whose *value* is extreme. These
rows are not extreme - they are **censored**: their recorded value is a bound, not a measurement. The
distinction matters because the fixes are opposite. You may sometimes drop an outlier. Dropping censored
rows deletes the evidence that the top of the range exists.

**"A lower RMSE always means a better model."** For the same target, the same rows and the same units,
lower is better. Change any of those and the comparison stops meaning anything - which is why the
"deleted them" row of the failure lab cannot be compared with the row above it on the ordinary rows alone.

**"Segment analysis is a fairness thing."** It is a *correctness* thing that happens to be how fairness
problems are found. Any model with a systematically bad segment is broken for that segment, whether or not
the segment corresponds to people.

**"We picked boosting, so linear regression was a waste of time."** The linear model is the reason we know
boosting's 0.4705 is worth having. Without it the number has no scale - and if boosting had come in at
0.74, the correct decision would have been to ship the linear model and save the complexity.

## Exercises

### Quick understanding

**E1.** The test RMSE (0.4574) came out *better* than the watch RMSE (0.4705). Give two explanations, and
say which one you would need more evidence to rule out.

**E2.** Why was the split made before the histograms were plotted, rather than after?

**E3.** In one sentence each, say what `fit`, `watch` and `test` are allowed to be used for.

### Hand calculation

**E4.** A model is scored on 500 rows. 450 of them have RMSE 0.40; the other 50 have RMSE 1.20. Compute
the overall RMSE.

**E5.** 490 of those 500 rows have RMSE 0.40. How bad would the remaining **10** have to be for the
overall RMSE across all 500 to reach 0.50?

**E6.** A segment of 200 rows has a mean error of +0.30 and an RMSE of 0.50. What is the standard
deviation of its errors? *(Hint: mean squared error = bias² + variance.)*

**E7.** Predictions 2.1, 3.4, 5.0 against actuals 1.9, 3.9, 5.0, where the third row is at the ceiling.
Compute RMSE with the third row and without it, and say which figure you would report.

### Coding - a worked example, then the same thing twice more

**E8 is done for you.** Read it, then do E9 with the gaps filled in, then E10 from nothing. This is
deliberate: copying a worked solution is how you learn the shape, and the value comes from the two after
it.

**E8 (worked below).** Segment the test set by whether `AveOccup` is above 6 - the institutional block
groups - and report rows, RMSE and mean error.

**E9 (fill the gaps).** Do the same for block groups where `AveRooms` is above 10. A skeleton is given
with three blanks.

**E10 (from nothing).** Do the same for the 10% of block groups with the highest `Population`. No
skeleton.

**E11.** Write a function `worst_segment(column, bins)` that cuts a column into bins, scores every bin,
and returns the name of the bin with the highest RMSE. Run it on all eight features and say which one
finds the biggest gap.

**E12.** Refit the chosen model on `fit` **and** `watch` together, then score it on the test set. Does it
improve? Explain why you would expect it to, and why doing this after seeing the test score would be
cheating.

### Interpretation

**E13.** The oldest housing band (46-52 years) has RMSE 0.5543 against 0.4094 for the 31-45 band. Give
two competing explanations and say what you would compute to tell them apart.

**E14.** The model's largest prediction anywhere on the test set is 5.4737, above the ceiling of 5.00001.
How is that possible, and is it a bug?

**E15.** The mean error over the whole test set is **+0.0079** - essentially zero - and a colleague
concludes the model is unbiased. Using the two rows of the ceiling table, show why that conclusion does
not follow.

### Debugging

**E16.** A colleague's notebook reports a test RMSE of 0.31. You find `train_test_split` called with no
`random_state`, inside a loop, with the best result kept. What number are they actually reporting?

**E17.** Another colleague standardises the features using `StandardScaler().fit(frame[FEATURES])` before
splitting. The scores barely change. Is the leak harmless?

### Exam and interview reasoning

**E18.** *"Your model has an RMSE of 0.46. Is that good?"* Answer it properly in under a minute.

**E19.** *"How would you know if your model was failing for a particular group of users?"* Answer, then
say what you would do if the group were 1% of the data.

### Transfer to a different situation

**E20.** A hospital's readmission model has an overall AUC of 0.82. Which segments would you insist on
seeing before it went live, and what would you do if one of them was small and bad?

**E21.** Your company's delivery-time model was trained on data where anything over 90 minutes was
recorded as "90+". Describe the failure this will produce and two ways to handle it.

### Explain it to someone non-technical

**E22.** Explain to a housing policy officer, in under 120 words, why the model should not be used to
value expensive neighbourhoods - without using the words *censored*, *bias* or *RMSE*.

### Optional challenge

**E23.** Treat the ceiling honestly: fit a model that predicts P(value is at the ceiling) and a second
that predicts the value for rows below it, then combine them. Does the combined model beat 0.4574 on the
ordinary rows? Does it give you anything better to say about the capped ones?

In [ ]:
# E8, worked. The pattern every segment answer in this chapter uses:
#   1. add a column that names the group each row belongs to
#   2. group by it
#   3. report rows, RMSE and mean error - never RMSE alone
scored["institutional"] = np.where(scored["AveOccup"] > 6, "AveOccup above 6", "ordinary")
print(by_segment("institutional").to_string(index=False, float_format=lambda v: "%.4f" % v))

**Read the `rows` column before the `RMSE` column - and this is why.** The institutional group scores
1.3059, nearly three times the overall error, which looks alarming until you notice it is **17 rows**.
Seventeen rows can produce almost any RMSE by luck; two unusual block groups would move it a long way.

**The honest answer here is "too few to say", and that is a finding rather than a failure.** Contrast it
with the ceiling segment: 181 rows, a clear direction to the bias, and a mechanism that explains it. One
of those two findings belongs in the report as a result; the other belongs in it as a question.

**Now E9.** The shape is identical; three blanks are marked `____`. Copy it into the empty cell below and
fill them in.

```python
# blank 1: the column to test
# blank 2: the threshold from the exercise
# blank 3: the name of the column you just created
scored["many rooms"] = np.where(scored[____] > ____, "AveRooms above 10", "ordinary")
print(by_segment(____).to_string(index=False, float_format=lambda v: "%.4f" % v))
```

In [ ]:
# E9. Paste the skeleton above and fill in the three blanks.

E10 has no skeleton at all. Segment by the top 10% of `Population` and report the same three numbers.

In [ ]:
# E10. Your turn, from nothing.

## Mastery check

You are ready for the module assessment if you can, without looking anything up:

1. State what each of `fit`, `watch` and `test` may be used for, and name the failure that follows from
   breaking each rule.
2. Look at a target histogram and say whether the variable is censored.
3. Write the pooling identity for RMSE and use it to explain why a small segment hides.
4. Produce a segment table with rows, RMSE and mean error, and say why all three columns are needed.
5. Explain why deleting inconvenient rows improved one number and damaged the thing that mattered.
6. Write the four-paragraph report above for a dataset you have not seen, in your own words.

## What should now feel instinctive

- **Split before looking.** Not as a rule you follow, as a thing you would feel uncomfortable not doing.
- **A baseline first, always** - the number that tells you whether anything else was worth it.
- **One metric is a summary, not a description.** Ask "for whom is this number wrong?" every time.
- **Bias and RMSE are different questions.** A segment can have near-zero RMSE and a terrible bias, or the
  reverse, and the two demand different fixes.
- **Look at the target's distribution before choosing a model**, because ceilings, floors and spikes at
  zero change what a model can possibly do.

## Flashcards

| Front | Back |
|---|---|
| Why split before exploring? | Anything you learn from looking becomes a decision made using that data |
| What is `watch` for? | Every choice - model, depth, rounds. Never for the final number |
| Censored vs outlier | Outlier: an extreme *value*. Censored: the value is a *bound*, the real one was never recorded |
| Pooling identity for MSE | `MSE_all = sum(n_g * MSE_g) / N`; RMSE is its square root |
| Why does a small bad segment hide? | It enters the pooled mean weighted by its size |
| Three columns of a segment table | rows, RMSE, mean error - size, magnitude, direction |
| Absolute error flat, relative error not | The same dollars are a bigger share of a smaller value |
| Effect of deleting awkward rows | The metric improves because the evidence is gone, not the problem |
| Forest with fit RMSE 0.19, watch 0.53 | Normal. A forest memorises its training data and still generalises |
| What a report must contain | What was measured, on what, where it fails, what it must not be used for |

## Next

**Module 05 is finished.** The assessment (`assessments/05_regression_assessment.ipynb`) is cumulative
across 05-01 to 05-12.

**06-01 opens classification**, where the target is a label rather than a number. Almost everything in
this chapter survives the change - split first, baseline first, segment the error - and one thing does
not: RMSE stops being available, and choosing what replaces it turns out to be most of the work.